In [1]:
!pip install torchmetrics
!pip install torchsummary
import os
import json
import random
from google.colab import drive,files
from IPython.display import clear_output
drive.mount('/content/drive')
clear_output()

In [ ]:
import numpy as np
import torch
from torchsummary import summary
from torchmetrics.classification import BinaryAccuracy
from torch import nn,optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import models
from torchvision import datasets,transforms
import matplotlib.pyplot as plt
import shutil # Для удаления папки
from wildFire_architecture import *
from Training_Wild_Fire_Models import *

In [3]:
shutil.rmtree('/content/drive/MyDrive/Wild_Fire_Proj')

In [ ]:
# создание словоря для json файла в котором хранится ключ kaggle
token = {'username':'...', 'key':'...'}

In [5]:
!mkdir -p ~/.kaggle
with open('/root/.kaggle/kaggle.json','w') as f:
  json.dump(token,f)

In [6]:
!chmod 600 ~/.kaggle/kaggle.json

# Новый раздел

In [36]:
#model.load_state_dict

# Новый раздел

In [ ]:
model_dict = {'ResNet50':model_res50,'My_CNN':my_model}

In [ ]:
# Валидация
for key,val in model_dict.items():
  early_stopper = EarlyStopping(path=proj_path,name_model=key)
  model = val
  model.load_state_dict(torch.load(early_stopper.path), strict=False)
  model.eval()
  with torch.no_grad():
    correct = 0
    total = 0
    progress_bar_valid = tqdm(valid_gen,desc=f'Valid {key}')
    for imagen,labels in progress_bar_valid:
      imagen,labels = imagen.to(device),labels.to(device)
      outputs = model(imagen)
      preds = (outputs > 0.5).float()
      total += labels.size(0)
      correct += (preds==total).sum().item()
    val_accuracy = 100*correct/total.round(4)

  print(f'{key} Valid Accuracy = {val_accuracy:.3f}%')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
for name,model in model_dict.items():
  print(f'Generating confusion matrix for {name}')


In [ ]:
# Проверка модели на тестовых данных

model = best_model.load_state_dict(torch.load(results_valid['Params']))
model.eval()
with torch.no_grad():
  corect_test = 0
  total_test = 0
  for imagen,labels in test_gen:
    imagen,labels = imagen.to(device),labels.to(device)
    outputs = model(imagen)
    preds = (outputs>0.5).float
    total_test += labels.size(0)
    corect_test += (preds==total_test).sum().item()
  print(f'Test Accuracy {100*corect_test/total_test:.4f}%')
